In [21]:
#Only good for static websites
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re

In [29]:
urls = ["https://catalog.ucdavis.edu/departments-programs-degrees/electrical-computer-engineering/electrical-engineering-bs/#requirementstext"
        ,"https://catalog.ucdavis.edu/departments-programs-degrees/electrical-computer-engineering/computer-engineering-bs/#requirementstext"
]

base_url = "https://catalog.ucdavis.edu"

df = pd.DataFrame(columns= ['Course', 'Title', 'Units', 'Course Description', 'Prerequisites', 'Concurrent Courses', 'Learning Activites', 'Credit Limitations', 'Grade Mode', 'General Education', 'Major Needed For'])



In [16]:
page = requests.get(urls[0])

soup = BeautifulSoup(page.text, 'html')

major_associated = ""

In [13]:
major_associated = soup.find('title')
major_associated = str(major_associated).replace("<title>General Catalog - ", "").replace(", Bachelor of Science</title>", "")

In [26]:
def get_course_info(links):

        for link in links:

            page = requests.get(link)

            soup = BeautifulSoup(page.text, 'html')

            block = soup.find('div', class_ = 'courseblock')

            titles = [t.get_text(strip=True).replace('—\xa0', '') for t in block.find_all('b')]
            #print(titles)

            first_section = block.find_all('p')
            second_section = block.find_all('li')

            print(second_section)

            if len(first_section) == 1:

                course_description = first_section[0].get_text(strip=True).replace('Course Description:', '').strip()
                #print(course_description)

                prereq_description = None
                concurrent_courses = None

            elif len(first_section) == 2:

                course_description = first_section[0].get_text(strip=True).replace('Course Description:', '').strip()
                #print(course_description)

                prereq_description = first_section[1].get_text(strip=True).replace('Prerequisite(s):', '').strip()
                #print(prereq_description)

                concurrent_courses = None
                match = re.findall(r'\b([A-Z]{2,4}\s*\d{2,3}[A-Z]?)\s*\(can be concurrent\)', prereq_description)
                if match:
                    concurrent_courses = ', '.join(match)




            learning_activities = second_section[0].get_text(strip=True).replace('Learning Activities:', '').strip()


            if len(second_section) == 3:

                credit_limitations = None


                grade_mode = second_section[1].get_text(strip=True).replace('Grade Mode:', '').strip()


                
                gen_ed = second_section[2].get_text(strip=True).replace('General Education:', '').strip()

            elif len(second_section) == 4:

                credit_limitations = second_section[1].get_text(strip=True).replace('Credit Limitation(s):', '').strip()


                grade_mode = second_section[2].get_text(strip=True).replace('Grade Mode:', '').strip()


                
                gen_ed = second_section[3].get_text(strip=True).replace('General Education:', '').strip()



            full_data = titles + [course_description, prereq_description, concurrent_courses, 
                              learning_activities, credit_limitations, grade_mode, gen_ed, major_associated]
            print(full_data)

            course_name = titles[0]
            length = len(df)

            existing_index = df.index[df['Course'] == course_name].tolist()

            if existing_index:
                idx = existing_index[0]
                existing_majors = df.at[idx, 'Major Needed For']
                if major_associated not in existing_majors:
                    df.at[idx, 'Major Needed For'] = f"{existing_majors}, {major_associated}"
            else:
                length = len(df)
                df.loc[length] = full_data

In [30]:

for url in urls:
    
    page = requests.get(url)

    soup = BeautifulSoup(page.text, 'html')

    table = soup.find('table')

    classes = table.find_all('a', class_='bubblelink code')

    links = [base_url + a['href'] for a in classes if 'href' in a.attrs]

    major_associated = soup.find('title')
    
    major_associated = str(major_associated).replace("<title>General Catalog - ", "").replace(", Bachelor of Science</title>", "")

    get_course_info(links)


[<li><span class="label"><em>Learning Activities:</em></span> Lecture 2 hour(s), Discussion 2 hour(s).</li>, <li><span class="label"><em>Grade Mode:</em></span> Letter.</li>, <li><span class="label"><em>General Education:</em></span> Arts &amp; Humanities (AH) or Social Sciences (SS); Oral Skills (OL).</li>]
['CMN 001', 'Introduction to Public Speaking', '(4 units)', 'Practice in the preparation and delivery of speeches based on principles and strategies of informing and persuading audiences drawn from the social sciences and humanities.', None, None, 'Lecture 2 hour(s), Discussion 2 hour(s).', None, 'Letter.', 'Arts & Humanities (AH) or Social Sciences (SS); Oral Skills (OL).', 'Electrical Engineering']
[<li><span class="label"><em>Learning Activities:</em></span> Web Virtual Lecture 2 hour(s), Web Electronic Discussion 2 hour(s).</li>, <li><span class="label"><em>Grade Mode:</em></span> Letter.</li>, <li><span class="label"><em>General Education:</em></span> Arts &amp; Humanities (AH

In [31]:
df

,Course,Title,Units,Course Description,Prerequisites,Concurrent Courses,Learning Activites,Credit Limitations,Grade Mode,General Education,Major Needed For
0,CMN 001,Introduction to Public Speaking,(4 units),Practice in the preparation and delivery of sp...,None,None,"Lecture 2 hour(s), Discussion 2 hour(s).",None,Letter.,Arts & Humanities (AH) or Social Sciences (SS)...,"Electrical Engineering, Computer Engineering"
1,CMN 001V,Introduction to Public Speaking,(4 units),Practice in the preparation and delivery of sp...,None,None,"Web Virtual Lecture 2 hour(s), Web Electronic ...",None,Letter.,Arts & Humanities (AH) or Social Sciences (SS)...,"Electrical Engineering, Computer Engineering"
2,ENG 003,Introduction to Engineering Design,(4 units),Introduction to the engineering design process...,Completion of Entry Level Writing Requirement ...,None,"Lecture 2 hour(s), Studio 2 hour(s), Project 2...",Enrollment Restriction(s):Pass One restricted ...,Letter.,Science & Engineering (SE) or Social Sciences ...,"Electrical Engineering, Computer Engineering"
3,ENG 003Y,Introduction to Engineering Design,(4 units),Introduction to the engineering design process...,Completion of Entry Level Writing Requirement ...,None,"Web Virtual Lecture 2 hour(s), Studio 2 hour(s...",Enrollment Restriction(s):Pass One restricted ...,Letter.,Science & Engineering (SE) or Social Sciences ...,"Electrical Engineering, Computer Engineering"
4,MAT 021A,Calculus,(4 units),"Functions, limits, continuity. Slope and deriv...","Two years of high school algebra, plane geomet...",None,"Lecture 3 hour(s), Discussion 1 hour(s).",Not open for credit to students who have compl...,Letter.,Science & Engineering (SE); Quantitative Liter...,"Electrical Engineering, Computer Engineering"
...,...,...,...,...,...,...,...,...,...,...,...
125,MGT 140,Marketing for the Technology-Based Enterprise,(4 units),"This version has ended; see updated course, be...",None,None,"Lecture 3 hour(s), Discussion 1 hour(s).",None,Letter.,Social Sciences (SS).,Computer Engineering
126,MGT 150,Technology Management,(4 units),"This version has ended; see updated course, be...",None,None,"Lecture 3 hour(s), Discussion 1 hour(s).",None,Letter.,Social Sciences (SS).,Computer Engineering
127,MGT 160,Financing New Business Ventures,(4 units),"This version has ended; see updated course, be...",(MGT 011AorMGT 011AVorMGT 011AY); (STA 013orST...,None,"Lecture 3 hour(s), Discussion 1 hour(s).",None,Letter.,Social Sciences (SS).,Computer Engineering
128,MGT 170,Management Accounting & Control,(4 units),"This version has ended; see updated course, be...",(MGT 011AorMGT 011AVorMGT 011AY);MGT 011B; or ...,None,"Lecture 3 hour(s), Discussion 1 hour(s).",None,Letter.,Social Sciences (SS).,Computer Engineering


In [32]:
df.to_excel('electrical_and_computer_engineering.xlsx', index=False)